# 00 — Setup and sample data

**LREC-COLING 2026 tutorial — LLM-as-annotator pipelines**

This notebook prepares a Colab-friendly workspace and creates a small multilingual toy dataset used by the following notebooks.

The examples are **pedagogical placeholders**, not an authoritative linguistic resource. Replace them with your project data before any real experiment.

In [ ]:
# In Colab, most packages below are already available. This keeps the notebooks portable.
!pip -q install jsonschema scikit-learn

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_DIR = Path('../lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
PROMPT_DIR = PROJECT_DIR / 'prompts'

for d in [DATA_DIR, SCHEMA_DIR, OUTPUT_DIR, PROMPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project directory:', PROJECT_DIR)

## Data model

Each row corresponds to one pre-tokenised sentence. The crucial rule is that later model outputs must contain **exactly one annotation per input token**.

Core columns:

- `id`: stable sentence identifier
- `language`: language name
- `script`: script/writing system
- `domain`: source genre or collection
- `text`: original sentence
- `tokens`: JSON-encoded list of input tokens
- `gold_pos`: JSON-encoded list of UPOS tags, when available
- `gold_lemma`: JSON-encoded list of lemmas, when available
- `gold_features`: JSON-encoded list of feature dictionaries
- `split`: `fewshot`, `eval`, or `unlabeled`
- `notes`: reminder that this sample is illustrative

In [ ]:
def j(x):
    return json.dumps(x, ensure_ascii=False)

rows = [
    {
        'id': 'grc_001', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy',
        'text': 'λόγος ἐστὶ καλός .',
        'tokens': ['λόγος', 'ἐστὶ', 'καλός', '.'],
        'gold_pos': ['NOUN', 'AUX', 'ADJ', 'PUNCT'],
        'gold_lemma': ['λόγος', 'εἰμί', 'καλός', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Pres'},
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'grc_002', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy',
        'text': 'οἱ ἄνδρες γράφουσι .',
        'tokens': ['οἱ', 'ἄνδρες', 'γράφουσι', '.'],
        'gold_pos': ['DET', 'NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['ὁ', 'ἀνήρ', 'γράφω', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur', 'Gender': 'Masc'},
            {'Case': 'Nom', 'Number': 'Plur', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Pres'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'grc_003', 'language': 'Ancient Greek', 'script': 'Greek', 'domain': 'toy_noisy',
        'text': 'βασιλεὺς εἶπεν λόγον .',
        'tokens': ['βασιλεὺς', 'εἶπεν', 'λόγον', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['βασιλεύς', 'λέγω', 'λόγος', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'xcl_001', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy',
        'text': 'այր մի եկն .',
        'tokens': ['այր', 'մի', 'եկն', '.'],
        'gold_pos': ['NOUN', 'NUM', 'VERB', 'PUNCT'],
        'gold_lemma': ['այր', 'մի', 'գալ', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'xcl_002', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy',
        'text': 'թագաւորն գրեաց նամակ .',
        'tokens': ['թագաւորն', 'գրեաց', 'նամակ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['թագաւոր', 'գրել', 'նամակ', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'xcl_003', 'language': 'Classical Armenian', 'script': 'Armenian', 'domain': 'toy_noisy',
        'text': 'աշակերտք ընթերցան գիրք .',
        'tokens': ['աշակերտք', 'ընթերցան', 'գիրք', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['աշակերտ', 'ընթեռնուլ', 'գիրք', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Past'},
            {'Case': 'Acc', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'oge_001', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy',
        'text': 'კაცი მოვიდა .',
        'tokens': ['კაცი', 'მოვიდა', '.'],
        'gold_pos': ['NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['კაცი', 'მოსვლა', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'oge_002', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy',
        'text': 'წიგნი კეთილი არს .',
        'tokens': ['წიგნი', 'კეთილი', 'არს', '.'],
        'gold_pos': ['NOUN', 'ADJ', 'AUX', 'PUNCT'],
        'gold_lemma': ['წიგნი', 'კეთილი', 'ყოფნა', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Case': 'Nom', 'Number': 'Sing'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Pres'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'oge_003', 'language': 'Old Georgian', 'script': 'Georgian', 'domain': 'toy_noisy',
        'text': 'მოწაფენი წერენ წიგნსა .',
        'tokens': ['მოწაფენი', 'წერენ', 'წიგნსა', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['მოწაფე', 'წერა', 'წიგნი', '.'],
        'gold_features': [
            {'Case': 'Nom', 'Number': 'Plur'},
            {'Person': '3', 'Number': 'Plur', 'Tense': 'Pres'},
            {'Case': 'Dat', 'Number': 'Sing'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'syr_001', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy',
        'text': 'ܓܒܪܐ ܐܬܐ .',
        'tokens': ['ܓܒܪܐ', 'ܐܬܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'PUNCT'],
        'gold_lemma': ['ܓܒܪܐ', 'ܐܬܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {}
        ],
        'split': 'fewshot'
    },
    {
        'id': 'syr_002', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy',
        'text': 'ܡܠܟܐ ܟܬܒ ܐܓܪܬܐ .',
        'tokens': ['ܡܠܟܐ', 'ܟܬܒ', 'ܐܓܪܬܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['ܡܠܟܐ', 'ܟܬܒ', 'ܐܓܪܬܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Number': 'Sing', 'Gender': 'Fem'},
            {}
        ],
        'split': 'eval'
    },
    {
        'id': 'syr_003', 'language': 'Syriac', 'script': 'Syriac', 'domain': 'toy_noisy',
        'text': 'ܬܠܡܝܕܐ ܩܪܐ ܟܬܒܐ .',
        'tokens': ['ܬܠܡܝܕܐ', 'ܩܪܐ', 'ܟܬܒܐ', '.'],
        'gold_pos': ['NOUN', 'VERB', 'NOUN', 'PUNCT'],
        'gold_lemma': ['ܬܠܡܝܕܐ', 'ܩܪܐ', 'ܟܬܒܐ', '.'],
        'gold_features': [
            {'Number': 'Sing', 'Gender': 'Masc'},
            {'Person': '3', 'Number': 'Sing', 'Tense': 'Past'},
            {'Number': 'Sing', 'Gender': 'Masc'},
            {}
        ],
        'split': 'eval'
    },
]

# Add metadata and serialise list columns for CSV.
for r in rows:
    r['notes'] = 'Toy pedagogical placeholder; replace with project-validated data.'
    for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
        r[col] = j(r[col])

df = pd.DataFrame(rows)
csv_path = DATA_DIR / 'toy_sentences.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'Wrote {len(df)} rows to {csv_path}')
df[['id', 'language', 'script', 'domain', 'text', 'split']]

In [ ]:
UPOS = [
    'ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN','NUM','PART','PRON','PROPN',
    'PUNCT','SCONJ','SYM','VERB','X'
]
FEATURES = ['Case', 'Number', 'Gender', 'Person', 'Tense', 'Mood', 'Voice']
CONFIDENCE = ['low', 'medium', 'high']

schema = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['sentence_id', 'language', 'tokens'],
    'properties': {
        'sentence_id': {'type': 'string'},
        'language': {'type': 'string'},
        'tokens': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['surface', 'lemma', 'upos', 'features', 'confidence', 'comment'],
                'properties': {
                    'surface': {'type': 'string'},
                    'lemma': {'type': ['string', 'null']},
                    'upos': {'type': 'string', 'enum': UPOS},
                    'features': {
                        'type': 'object',
                        'additionalProperties': {'type': ['string', 'null']}
                    },
                    'confidence': {'type': 'string', 'enum': CONFIDENCE},
                    'comment': {'type': ['string', 'null']}
                }
            }
        }
    }
}

schema_path = SCHEMA_DIR / 'pos_lemma_morph_schema.json'
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
print('Wrote schema to', schema_path)

## Next notebook

Continue with `01_prompting_zero_few_shot.ipynb`.

# 01 — Prompting: zero-shot and few-shot

This notebook builds prompts for token-level linguistic annotation. It can either call an API through a user-provided adapter or use a deterministic fallback that mimics realistic successes and failures.

Default mode is `USE_API = False`, so the notebook runs in Colab without secrets.

In [ ]:
!pip -q install jsonschema

In [ ]:
from pathlib import Path
import os, json, re, random, hashlib, datetime
import pandas as pd
import numpy as np

PROJECT_DIR = Path('../lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
for d in [DATA_DIR, SCHEMA_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

USE_API = False   # Change to True only after implementing/providing a safe provider adapter.
MODEL_NAME = 'fallback-mock-annotator-v0'
PROMPT_VERSION = 'v0.1'

print('USE_API =', USE_API)

## Load data

Run `00_setup_and_data.ipynb` first. If the data are missing, the next cell raises a clear error.

In [ ]:
csv_path = DATA_DIR / 'toy_sentences.csv'
if not csv_path.exists():
    raise FileNotFoundError('Run 00_setup_and_data.ipynb first to create toy_sentences.csv')

df = pd.read_csv(csv_path)
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    df[col] = df[col].apply(json.loads)

df[['id','language','script','domain','text','split']]

In [ ]:
UPOS = ['ADJ','ADP','ADV','AUX','CCONJ','DET','INTJ','NOUN','NUM','PART','PRON','PROPN','PUNCT','SCONJ','SYM','VERB','X']
FEATURES = ['Case', 'Number', 'Gender', 'Person', 'Tense', 'Mood', 'Voice']
CONFIDENCE = ['low', 'medium', 'high']

schema = json.loads((SCHEMA_DIR / 'pos_lemma_morph_schema.json').read_text(encoding='utf-8'))

## Prompt builder

The prompt fixes the task, the tokenisation policy, the allowed labels, and the expected JSON shape. Few-shot examples are selected from rows marked `split == "fewshot"`.

In [ ]:
def format_example(row):
    token_objs = []
    for surface, lemma, upos, feats in zip(row['tokens'], row['gold_lemma'], row['gold_pos'], row['gold_features']):
        token_objs.append({
            'surface': surface,
            'lemma': lemma,
            'upos': upos,
            'features': feats,
            'confidence': 'high',
            'comment': None
        })
    return json.dumps({
        'input': {'language': row['language'], 'sentence_id': row['id'], 'tokens': row['tokens']},
        'output': {'sentence_id': row['id'], 'language': row['language'], 'tokens': token_objs}
    }, ensure_ascii=False, indent=2)

fewshot_rows = df[df['split'] == 'fewshot'].copy()

def build_prompt(row, mode='zero_shot', max_examples=4):
    inventory = ', '.join(UPOS)
    prompt = f"""
You are assisting with linguistic annotation of historical and under-resourced languages.
Your task is to annotate the provided sentence token by token.

Return only valid JSON.
Do not translate the sentence.
Do not add, remove, split, merge, reorder, transliterate, or normalise tokens.
Use the provided UPOS inventory only: {inventory}.
Use feature names only from: {', '.join(FEATURES)}.
If unsure, keep the original token, use the best label you can, set confidence to "low", and add a short local comment.

Language: {row['language']}
Sentence ID: {row['id']}
Tokens: {json.dumps(row['tokens'], ensure_ascii=False)}
""".strip()
    if mode == 'few_shot':
        examples = []
        # Prefer same language example; then add others if needed.
        same_lang = fewshot_rows[fewshot_rows['language'] == row['language']]
        chosen = pd.concat([same_lang, fewshot_rows]).drop_duplicates('id').head(max_examples)
        for _, ex in chosen.iterrows():
            examples.append(format_example(ex))
        prompt += "\n\nFollow these validated examples:\n" + "\n\n".join(examples)
    prompt += "\n\nExpected JSON schema summary: sentence_id, language, tokens[{surface, lemma, upos, features, confidence, comment}]."
    return prompt

sample_row = df[df['split'] == 'eval'].iloc[0]
print(build_prompt(sample_row, mode='few_shot')[:2200])

## Provider adapter

The default tutorial path uses fallback predictions. If you want to call a model, implement `call_llm_api(prompt, schema)` for your chosen provider.

Keep the API key in an environment variable or Colab secret; never hard-code it in the notebook.

In [ ]:
def call_llm_api(prompt, schema=None):
    """Provider adapter placeholder.

    Implement this function for your chosen provider. It should return a raw string.
    Do not place API keys directly in the notebook.
    """
    raise NotImplementedError(
        'API mode is intentionally left as an adapter. Set USE_API=False or implement call_llm_api().'
    )

## Fallback annotator

This deterministic mock annotator is not a model. It creates plausible predictions with a few controlled failures so the validation and evaluation notebooks have something to analyse.

In [ ]:
def mock_prediction(row, mode='zero_shot'):
    tokens = list(row['tokens'])
    gold_pos = list(row['gold_pos'])
    gold_lemma = list(row['gold_lemma'])
    gold_features = list(row['gold_features'])

    out_tokens = []
    for i, surface in enumerate(tokens):
        upos = gold_pos[i]
        lemma = gold_lemma[i]
        feats = dict(gold_features[i])
        confidence = 'high'
        comment = None

        # Deterministic but interpretable perturbations.
        key = f"{row['id']}::{i}::{mode}"
        h = int(hashlib.md5(key.encode('utf-8')).hexdigest(), 16) % 100
        if mode == 'zero_shot':
            if h < 12 and upos == 'NOUN':
                upos = 'PROPN'
                confidence = 'medium'
                comment = 'Possible proper/common noun confusion.'
            elif h < 18 and upos == 'VERB':
                feats.pop('Tense', None)
                confidence = 'low'
                comment = 'Uncertain verbal morphology.'
            elif h < 22 and upos == 'DET':
                upos = 'ARTICLE'  # invalid on purpose
                confidence = 'medium'
                comment = 'Invalid label deliberately introduced.'
        else:  # few-shot: fewer errors
            if h < 6 and upos == 'NOUN':
                feats.pop('Case', None)
                confidence = 'medium'
                comment = 'Case uncertain.'

        out_tokens.append({
            'surface': surface,
            'lemma': lemma,
            'upos': upos,
            'features': feats,
            'confidence': confidence,
            'comment': comment
        })

    # One alignment failure in zero-shot mode.
    if mode == 'zero_shot' and row['id'] == 'xcl_002':
        out_tokens = out_tokens[:-1]

    pred = {'sentence_id': row['id'], 'language': row['language'], 'tokens': out_tokens}
    raw = json.dumps(pred, ensure_ascii=False)

    # One fenced response to test parser robustness.
    if mode == 'zero_shot' and row['id'] == 'oge_002':
        raw = '```json\n' + raw + '\n```'
    return raw

In [ ]:
def annotate_batch(data, mode='zero_shot'):
    records = []
    for _, row in data.iterrows():
        prompt = build_prompt(row, mode='few_shot' if mode == 'few_shot' else 'zero_shot')
        if USE_API:
            raw = call_llm_api(prompt, schema=schema)
            model_name = MODEL_NAME
        else:
            raw = mock_prediction(row, mode=mode)
            model_name = MODEL_NAME
        records.append({
            'sentence_id': row['id'],
            'language': row['language'],
            'split': row['split'],
            'mode': mode,
            'model_name': model_name,
            'prompt_version': PROMPT_VERSION,
            'prompt': prompt,
            'raw_response': raw,
            'timestamp': datetime.datetime.utcnow().isoformat() + 'Z'
        })
    return pd.DataFrame(records)

eval_df = df[df['split'].isin(['eval','unlabeled'])].copy()
zero_pred = annotate_batch(eval_df, mode='zero_shot')
few_pred = annotate_batch(eval_df, mode='few_shot')

for name, pred in [('zero_shot_raw.jsonl', zero_pred), ('few_shot_raw.jsonl', few_pred)]:
    path = OUTPUT_DIR / name
    pred.to_json(path, orient='records', lines=True, force_ascii=False)
    print('Wrote', path, len(pred), 'records')

zero_pred[['sentence_id','language','mode','raw_response']].head(3)

## Participant TODO

Change one part of the prompt and rerun only a small batch. Examples:

- add a stricter instruction about token alignment;
- add a few-shot example from the same language;
- require `confidence = low` when morphology is uncertain;
- remove the comments and see whether validation becomes easier.

Continue with `02_structured_outputs_and_validation.ipynb`.

# 02 — Structured outputs and validation

This notebook parses raw model responses and validates them before any linguistic evaluation.

Validation is split into:

1. JSON parsing
2. schema validation
3. token alignment
4. inventory checks

In [ ]:
!pip -q install jsonschema

In [ ]:
from pathlib import Path
import json, re
import pandas as pd
from jsonschema import Draft202012Validator

PROJECT_DIR = Path('../lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
SCHEMA_DIR = PROJECT_DIR / 'schemas'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

for d in [OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

schema = json.loads((SCHEMA_DIR / 'pos_lemma_morph_schema.json').read_text(encoding='utf-8'))
validator = Draft202012Validator(schema)

df = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    df[col] = df[col].apply(json.loads)
input_by_id = df.set_index('id').to_dict(orient='index')

In [ ]:
raw_files = [OUTPUT_DIR / 'zero_shot_raw.jsonl', OUTPUT_DIR / 'few_shot_raw.jsonl']
missing = [p for p in raw_files if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing}. Run 01_prompting_zero_few_shot.ipynb first.')
raw = pd.concat([pd.read_json(p, lines=True) for p in raw_files], ignore_index=True)
raw[['sentence_id','mode','raw_response']].head()

## Parser

The parser should be conservative. It can handle simple Markdown code fences, but it should not silently rewrite the model output into a different annotation.

In [ ]:
def extract_json_text(raw_text):
    text = str(raw_text).strip()
    # Remove a simple fenced code block if the entire response is fenced.
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()
    return text

def parse_response(raw_text):
    text = extract_json_text(raw_text)
    try:
        return json.loads(text), None
    except Exception as e:
        return None, f'parse_error: {type(e).__name__}: {e}'

parsed = []
for _, rec in raw.iterrows():
    obj, err = parse_response(rec['raw_response'])
    parsed.append({**rec.to_dict(), 'parsed': obj, 'parse_error': err})
parsed = pd.DataFrame(parsed)
parsed[['sentence_id','mode','parse_error']]

## Validators

A prediction may be valid JSON but still unusable because tokens are missing, surfaces changed, or labels are outside the inventory.

In [ ]:
def validate_prediction(rec):
    errors = []
    obj = rec['parsed']
    sentence_id = rec['sentence_id']

    if obj is None:
        return False, [rec['parse_error']]

    # JSON Schema validation.
    for err in sorted(validator.iter_errors(obj), key=lambda e: list(e.path)):
        path = '.'.join(str(p) for p in err.path)
        errors.append(f'schema_error at {path or "<root>"}: {err.message}')

    # Token alignment validation.
    input_row = input_by_id.get(sentence_id)
    if input_row is None:
        errors.append('unknown_sentence_id')
    else:
        input_tokens = input_row['tokens']
        output_tokens = obj.get('tokens', []) if isinstance(obj, dict) else []
        if len(output_tokens) != len(input_tokens):
            errors.append(f'token_count_mismatch: input={len(input_tokens)} output={len(output_tokens)}')
        else:
            for i, (inp, out_tok) in enumerate(zip(input_tokens, output_tokens)):
                surface = out_tok.get('surface') if isinstance(out_tok, dict) else None
                if surface != inp:
                    errors.append(f'surface_mismatch[{i}]: input={inp!r} output={surface!r}')

    return len(errors) == 0, errors

validated_records = []
for _, rec in parsed.iterrows():
    ok, errors = validate_prediction(rec)
    d = rec.to_dict()
    d['is_valid'] = ok
    d['validation_errors'] = errors
    validated_records.append(d)

validated = pd.DataFrame(validated_records)
validated[['sentence_id','mode','is_valid','validation_errors']]

In [ ]:
valid_path = OUTPUT_DIR / 'validated_predictions.jsonl'
invalid_path = OUTPUT_DIR / 'invalid_outputs.csv'

# JSONL cannot directly serialise nested objects from pandas unless we keep them JSON-compatible.
validated.to_json(valid_path, orient='records', lines=True, force_ascii=False)
validated[~validated['is_valid']][['sentence_id','mode','validation_errors','raw_response']].to_csv(invalid_path, index=False, encoding='utf-8')

print('Wrote', valid_path)
print('Wrote', invalid_path)
validated.groupby(['mode','is_valid']).size().reset_index(name='n')

## Inspect invalid outputs

Invalid outputs are not just technical noise. The invalid-output rate belongs in the method section.

In [ ]:
invalid = validated[~validated['is_valid']]
if len(invalid) == 0:
    print('No invalid outputs.')
else:
    display(invalid[['sentence_id','language','mode','validation_errors','raw_response']])

## Participant TODO

Pick one invalid output. Decide whether the fix should be:

- prompt change;
- schema change;
- preprocessing/tokenisation change;
- expert review;
- no automatic fix, only reporting.

Continue with `03_evaluation_and_error_analysis.ipynb`.

# 03 — Evaluation and error analysis

This notebook evaluates validated predictions against the toy gold standard.

It deliberately keeps format/alignment validity separate from linguistic quality.

In [ ]:
!pip -q install scikit-learn matplotlib

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

PROJECT_DIR = Path('../lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

validated_path = OUTPUT_DIR / 'validated_predictions.jsonl'
if not validated_path.exists():
    raise FileNotFoundError('Run 02_structured_outputs_and_validation.ipynb first.')

validated = pd.read_json(validated_path, lines=True)
gold = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    gold[col] = gold[col].apply(json.loads)

gold_by_id = gold.set_index('id').to_dict(orient='index')
validated[['sentence_id','mode','is_valid']].head()

## Flatten token-level predictions

Only valid, aligned outputs are used for linguistic scores. Invalid outputs remain counted separately.

In [ ]:
def flatten_predictions(validated_df):
    rows = []
    for _, rec in validated_df.iterrows():
        sid = rec['sentence_id']
        g = gold_by_id[sid]
        if not rec['is_valid']:
            continue
        parsed = rec['parsed']
        for i, pred_tok in enumerate(parsed['tokens']):
            rows.append({
                'sentence_id': sid,
                'language': rec['language'],
                'mode': rec['mode'],
                'token_index': i,
                'surface': g['tokens'][i],
                'gold_pos': g['gold_pos'][i],
                'pred_pos': pred_tok.get('upos'),
                'gold_lemma': g['gold_lemma'][i],
                'pred_lemma': pred_tok.get('lemma'),
                'gold_features': g['gold_features'][i],
                'pred_features': pred_tok.get('features', {}),
                'confidence': pred_tok.get('confidence'),
                'comment': pred_tok.get('comment')
            })
    return pd.DataFrame(rows)

tok = flatten_predictions(validated)
tok.head(10)

In [ ]:
validity_summary = validated.groupby(['mode','is_valid']).size().reset_index(name='n')
validity_summary

## POS and lemma scores

In [ ]:
def safe_mean(x):
    return float(np.mean(x)) if len(x) else np.nan

summary = []
for mode, sub in tok.groupby('mode'):
    summary.append({
        'mode': mode,
        'n_tokens_evaluated': len(sub),
        'pos_accuracy': safe_mean(sub['gold_pos'] == sub['pred_pos']),
        'lemma_exact_match': safe_mean(sub['gold_lemma'] == sub['pred_lemma']),
    })
summary = pd.DataFrame(summary)
summary

In [ ]:
by_lang = tok.assign(
    pos_correct=lambda x: x['gold_pos'] == x['pred_pos'],
    lemma_correct=lambda x: x['gold_lemma'] == x['pred_lemma']
).groupby(['mode','language']).agg(
    n_tokens=('surface','size'),
    pos_accuracy=('pos_correct','mean'),
    lemma_exact_match=('lemma_correct','mean')
).reset_index()
by_lang

## Morphological feature F1

We score feature-value pairs. For each token, compare sets like `Case=Nom` and `Number=Sing`.

In [ ]:
def feature_set(d):
    if not isinstance(d, dict):
        return set()
    return {f'{k}={v}' for k, v in d.items() if v not in [None, '', 'None']}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f1

rows = []
for mode, sub in tok.groupby('mode'):
    tp = fp = fn = 0
    for _, row in sub.iterrows():
        g = feature_set(row['gold_features'])
        p = feature_set(row['pred_features'])
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    prec, rec, f1 = prf(tp, fp, fn)
    rows.append({'mode': mode, 'feature_precision': prec, 'feature_recall': rec, 'feature_f1': f1, 'tp': tp, 'fp': fp, 'fn': fn})
feat_summary = pd.DataFrame(rows)
feat_summary

## Confusion matrix

In [ ]:
mode_to_plot = 'zero_shot'
sub = tok[tok['mode'] == mode_to_plot]
labels = sorted(set(sub['gold_pos']) | set(sub['pred_pos']))
cm = confusion_matrix(sub['gold_pos'], sub['pred_pos'], labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Gold')
ax.set_title(f'POS confusion matrix — {mode_to_plot}')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
fig.tight_layout()
fig_path = OUTPUT_DIR / f'pos_confusion_{mode_to_plot}.png'
fig.savefig(fig_path, dpi=160)
print('Saved', fig_path)
plt.show()

## Token-level error table

In [ ]:
def classify_error(row):
    if row['gold_pos'] != row['pred_pos']:
        return 'pos'
    if row['gold_lemma'] != row['pred_lemma']:
        return 'lemma'
    if feature_set(row['gold_features']) != feature_set(row['pred_features']):
        return 'morphology'
    return 'correct'

errors = tok.copy()
errors['error_type'] = errors.apply(classify_error, axis=1)
errors = errors[errors['error_type'] != 'correct'].copy()
errors[['sentence_id','language','mode','surface','gold_pos','pred_pos','gold_lemma','pred_lemma','error_type','confidence','comment']]

In [ ]:
summary_path = OUTPUT_DIR / 'summary_metrics.csv'
errors_path = OUTPUT_DIR / 'token_errors.csv'

summary.merge(feat_summary, on='mode').to_csv(summary_path, index=False, encoding='utf-8')
errors.to_csv(errors_path, index=False, encoding='utf-8')
print('Wrote', summary_path)
print('Wrote', errors_path)

## Participant TODO

Choose one error. Decide whether it is:

- a true linguistic error;
- a convention mismatch;
- caused by the prompt;
- caused by an unclear tagset;
- caused by an input/tokenisation issue.

Continue with `04_sampling_and_bootstrapping.ipynb`.

# 04 — Sampling and bootstrapping

This notebook builds a small review batch: examples that should be prioritised for expert review.

The point is not just to ask "What is the score?" but "What should we annotate next?".

In [ ]:
!pip -q install scikit-learn

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_DIR = Path('../lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

validated = pd.read_json(OUTPUT_DIR / 'validated_predictions.jsonl', lines=True)
gold = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens','gold_pos','gold_lemma','gold_features']:
    gold[col] = gold[col].apply(json.loads)

gold[['id','language','domain','text','split']]

## Sentence-level signals

We combine several signals:

- validation failure
- low confidence
- zero-shot vs few-shot disagreement
- use of fallback/unknown labels
- diversity across the corpus
- random slice for coverage

In [ ]:
def sentence_confidence_score(parsed):
    # Higher = more uncertain.
    if not isinstance(parsed, dict) or 'tokens' not in parsed:
        return 1.0
    mapping = {'high': 0.0, 'medium': 0.5, 'low': 1.0}
    vals = [mapping.get(tok.get('confidence'), 0.75) for tok in parsed.get('tokens', []) if isinstance(tok, dict)]
    return float(np.mean(vals)) if vals else 1.0

def pos_sequence(parsed):
    if not isinstance(parsed, dict) or 'tokens' not in parsed:
        return []
    return [tok.get('upos') for tok in parsed['tokens'] if isinstance(tok, dict)]

# Pivot zero/few predictions by sentence.
records = []
for _, rec in validated.iterrows():
    records.append({
        'sentence_id': rec['sentence_id'],
        'language': rec['language'],
        'mode': rec['mode'],
        'is_valid': bool(rec['is_valid']),
        'parsed': rec['parsed'],
        'uncertainty': sentence_confidence_score(rec['parsed']),
        'pos_sequence': pos_sequence(rec['parsed']),
        'validation_errors': rec['validation_errors']
    })
sig_long = pd.DataFrame(records)
sig = sig_long.pivot_table(index='sentence_id', columns='mode', values='uncertainty', aggfunc='first').reset_index()
sig.columns.name = None
sig = sig.rename(columns={'zero_shot': 'uncertainty_zero', 'few_shot': 'uncertainty_few'})
sig

In [ ]:
# Disagreement score between zero-shot and few-shot POS sequences.
def disagreement_for_sentence(sid):
    sub = sig_long[sig_long['sentence_id'] == sid]
    seqs = {row['mode']: row['pos_sequence'] for _, row in sub.iterrows()}
    z, f = seqs.get('zero_shot', []), seqs.get('few_shot', [])
    if not z or not f or len(z) != len(f):
        return 1.0
    return float(np.mean([a != b for a, b in zip(z, f)]))

sig['disagreement'] = sig['sentence_id'].apply(disagreement_for_sentence)

valid_fail = sig_long.groupby('sentence_id')['is_valid'].apply(lambda x: 1.0 - float(all(x))).reset_index(name='validation_failure')
sig = sig.merge(valid_fail, on='sentence_id', how='left')
sig['uncertainty'] = sig[['uncertainty_zero','uncertainty_few']].mean(axis=1)
sig

## Diversity score

Here we use a simple character n-gram TF-IDF representation of the sentence text. In a real project, you might use embeddings and cluster-based sampling.

In [ ]:
meta = gold.rename(columns={'id': 'sentence_id'})[['sentence_id','language','domain','text','tokens']]
sig = sig.merge(meta, on='sentence_id', how='left')

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X = vectorizer.fit_transform(sig['text'].fillna(''))
sim = cosine_similarity(X)
# Diversity proxy: 1 - average similarity to all other examples.
if len(sig) > 1:
    avg_sim = (sim.sum(axis=1) - 1) / (len(sig) - 1)
else:
    avg_sim = np.zeros(len(sig))
sig['diversity'] = 1 - avg_sim
sig[['sentence_id','language','text','diversity']]

## Hybrid priority score

Weights are placeholders. The point is to make the selection rule explicit and auditable.

In [ ]:
# Normalize helper.
def normalize(s):
    s = pd.Series(s).astype(float)
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

sig['priority'] = (
    0.35 * normalize(sig['uncertainty']) +
    0.30 * normalize(sig['disagreement']) +
    0.20 * normalize(sig['diversity']) +
    0.15 * normalize(sig['validation_failure'])
)

sig = sig.sort_values('priority', ascending=False)
sig[['sentence_id','language','domain','uncertainty','disagreement','diversity','validation_failure','priority']]

## Build a stratified review batch

We select high-priority cases while keeping language coverage. For a real project, also reserve some random examples.

In [ ]:
K_PER_LANGUAGE = 1
review_batch = (
    sig.sort_values(['language','priority'], ascending=[True, False])
       .groupby('language', group_keys=False)
       .head(K_PER_LANGUAGE)
       .sort_values('priority', ascending=False)
       .copy()
)

review_batch['selection_reason'] = review_batch.apply(
    lambda r: f"priority={r['priority']:.2f}; uncertainty={r['uncertainty']:.2f}; disagreement={r['disagreement']:.2f}; validation_failure={r['validation_failure']:.0f}",
    axis=1
)

review_path = OUTPUT_DIR / 'review_batch.csv'
review_batch.to_csv(review_path, index=False, encoding='utf-8')
print('Wrote', review_path)
review_batch[['sentence_id','language','text','selection_reason']]

## Simulating the bootstrapping update

After expert review, corrected examples can become:

- new gold examples;
- few-shot examples;
- error cases in the guideline;
- training data for a specialised model.

In [ ]:
# Placeholder for expert corrections.
# In a real workshop, participants would fill these columns manually or via a spreadsheet.
expert_template = review_batch[['sentence_id','language','text','tokens','selection_reason']].copy()
expert_template['expert_checked'] = False
expert_template['expert_notes'] = ''
expert_template_path = OUTPUT_DIR / 'expert_review_template.csv'
expert_template.to_csv(expert_template_path, index=False, encoding='utf-8')
print('Wrote', expert_template_path)
expert_template

## Participant TODO

Open `review_batch.csv` or `expert_review_template.csv` and decide whether the selected examples look genuinely useful.

Would you change the weights? Would you add a random slice? Would you stratify by domain instead of language?

## End of notebooks

You now have a full minimal workflow:

1. create/load data;
2. prompt zero/few-shot;
3. parse and validate;
4. evaluate and analyse errors;
5. select examples for expert review.